# Gaia DR3 XP derivative feature augmentation (110D)

This notebook augments an existing **classification/features CSV** (must contain `source_id`) with an additional
**110-dimensional derivative feature vector** derived from **Gaia DR3 XP continuous mean spectra**.

## Concept

For each source:
1. Load XP continuous coefficients for BP and RP (55 + 55).
2. Convert coefficients to sampled flux on a fixed grid using GaiaXPy's basis conversion:

   $$
   f(u) \approx A\,c
   $$
   where `c` are XP coefficients and `A` is the design matrix at sampling grid points.
3. Compute the spectral derivative on the grid (via finite differences).
4. Project the derivative flux back to coefficient space using a ridge-stabilized pseudo-inverse:

   $$
   c' \approx f'\,(A^T A + \lambda I)^{-1} A^T
   $$
5. Concatenate BP and RP derivative coefficients into 110D, then L2-normalize per source.

## Inputs
- `GAIA_AIP_TOKEN` for Gaia@AIP access.
- A CSV file "binary_vosa_XPcoeff_110d_L2.csv" from xpcoeff_feature_build_binary.ipynb.

## Output
- New CSV with extra columns `d000..d109` (`float32`), saved to `out_data/`.


In [ ]:
import io
import os
import time
import random
import ast
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
from astropy.io.votable import parse_single_table

from gaiaxpy import convert

## Configuration (auth, paths, and main parameters)

In [ ]:
# --- AUTH ---
GAIA_AIP_TOKEN = os.getenv("GAIA_AIP_TOKEN", "Token <YOUR_TOKEN_GOES_HERE>").strip()
#                                                    ^^^^^^^^^^^^^^^^^^^^
if not GAIA_AIP_TOKEN.startswith("Token "):
    GAIA_AIP_TOKEN = "Token " + GAIA_AIP_TOKEN

TAP_URL = "https://gaia.aip.de/tap"

sess = requests.Session()
sess.headers["Authorization"] = GAIA_AIP_TOKEN

# Input classification/features CSV (must contain source_id)
print("CWD:", Path.cwd())

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "out_data"
if not OUT_DIR.exists():
    hits = list(ROOT.rglob("out_data"))
    if hits:
        OUT_DIR = hits[0]

print("ROOT   :", ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR :", OUT_DIR)

# Default file name (change if needed)
CLASSIFICATION_NAME = "binary_vosa_XPcoeff_110d_L2.csv"
CLASSIFICATION_CSV  = OUT_DIR / CLASSIFICATION_NAME

if not CLASSIFICATION_CSV.exists():
    hits = list(ROOT.rglob(CLASSIFICATION_NAME))
    if hits:
        CLASSIFICATION_CSV = hits[0]

if not CLASSIFICATION_CSV.exists():
    raise FileNotFoundError(
        f"Missing classification/features CSV: {CLASSIFICATION_CSV}\n"
        f"Tried OUT_DIR/{CLASSIFICATION_NAME} and ROOT.rglob('{CLASSIFICATION_NAME}').\n"
        "Either place the file in out_data/ or update CLASSIFICATION_NAME."
    )

print("CLASSIFICATION_CSV:", CLASSIFICATION_CSV)

SOURCE_ID_COL = "source_id"
LABEL_COL = "y"

# --- XP processing ---
SAMPLING  = np.linspace(0.0, 60.0, 600)
RIDGE_LAM = 1e-3

# --- Query chunking ---
ID_CHUNK_SIZE = 500

# --- Output ---
MERGED_CSV = OUT_DIR / CLASSIFICATION_NAME.replace(".csv", "_w/derivatives.csv")
print("MERGED_CSV =", MERGED_CSV)

## Load source IDs from the classification/features CSV

In [ ]:
df_cls = pd.read_csv(CLASSIFICATION_CSV)

if SOURCE_ID_COL not in df_cls.columns:
    raise KeyError(f"Column '{SOURCE_ID_COL}' not found in {CLASSIFICATION_CSV}. Columns: {list(df_cls.columns)}")

source_ids = pd.to_numeric(df_cls[SOURCE_ID_COL], errors="coerce").dropna().astype("int64").unique()
source_ids = np.sort(source_ids)

print("Classification rows:", len(df_cls))
print("Unique source_ids:", len(source_ids))
if len(source_ids) > 0:
    print("First/last:", int(source_ids[0]), int(source_ids[-1]))

## TAP (async) helpers

In [13]:
def tap_async_start(
    session: requests.Session,
    query: str,
    *,
    lang: str = "ADQL",
    runid: str = "xp_ids",
    max_retries: int = 5,
    backoff_s: float = 2.0,
):
    url = f"{TAP_URL}/async"
    q = query.strip().rstrip(";")
    payload = {"REQUEST":"doQuery", "LANG":lang, "FORMAT":"votable", "QUERY":q, "RUNID":runid}

    last_err = None
    for attempt in range(max_retries + 1):
        try:
            r = session.post(url, data=payload, allow_redirects=False, timeout=120)
        except requests.RequestException as e:
            last_err = e
            if attempt < max_retries:
                sleep_s = backoff_s * (2 ** attempt)
                sleep_s *= 0.5 + random.random()  # jitter
                time.sleep(sleep_s)
                continue
            raise

        if r.status_code in (302, 303) and "Location" in r.headers:
            job_url = r.headers["Location"]
            if job_url.startswith("/"):
                job_url = "https://gaia.aip.de" + job_url
            session.post(f"{job_url}/phase", data={"PHASE":"RUN"}, timeout=60).raise_for_status()
            return job_url

        # retry on transient gateway/load errors
        if r.status_code in (429, 500, 502, 503, 504):
            last_err = RuntimeError(f"TAP async start failed: HTTP {r.status_code}; body={r.text[:300]!r}")
            if attempt < max_retries:
                sleep_s = backoff_s * (2 ** attempt)
                sleep_s *= 0.5 + random.random()
                time.sleep(sleep_s)
                continue

        raise RuntimeError(f"TAP async start failed: HTTP {r.status_code}; body={r.text[:300]!r}")

    raise last_err if last_err is not None else RuntimeError("TAP async start failed (unknown).")

def tap_async_wait(session: requests.Session, job_url: str, *, timeout_s: int = 240):
    t0 = time.time()
    while True:
        ph = session.get(f"{job_url}/phase", timeout=60).text.strip()
        if ph in ("COMPLETED", "ERROR", "ABORTED"):
            break
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"TAP async timed out (> {timeout_s}s)")
        time.sleep(1.5)

    if ph != "COMPLETED":
        err = session.get(f"{job_url}/error", timeout=60).text if ph == "ERROR" else ""
        raise RuntimeError(f"TAP async ended with phase={ph}. {err[:800]}")
    return ph

def tap_async_download_votable(session: requests.Session, job_url: str) -> pd.DataFrame:
    res_url = f"{job_url}/results/result"
    content = session.get(res_url, timeout=300).content
    tab = parse_single_table(io.BytesIO(content)).to_table(use_names_over_ids=True)
    return tab.to_pandas()

## Robust parsing of array-like columns

In [14]:
def to_1d_float(x, expected_len: Optional[int] = None) -> np.ndarray:
    # If the column is a masked array, drop the mask and keep numeric data
    if isinstance(x, np.ma.MaskedArray):
        x = np.ma.getdata(x)

    # Parse string representations if TAP returns arrays as strings
    if isinstance(x, str):
        s = x.strip()
        try:
            x = ast.literal_eval(s)
        except Exception:
            s = s.strip("[]()")
            parts = re.split(r"[,\s]+", s)
            parts = [p for p in parts if p]
            x = [float(p) for p in parts]

    arr = np.asarray(x, dtype=float).reshape(-1)
    if expected_len is not None and arr.size != expected_len:
        raise ValueError(f"Expected len={expected_len}, got {arr.size}")
    return arr

## Build GaiaXPy design matrices (cached per basis-id pair)

In [15]:
def build_design_matrices_via_convert(bp_basis_id: int, rp_basis_id: int, sampling: np.ndarray):
    n = 55
    I = np.eye(n, dtype=float)

    # Minimal columns for gaiaxpy input validation (kept lightweight)
    common = {
        "bp_n_parameters": n,
        "bp_standard_deviation": 1.0,
        "rp_n_parameters": n,
        "rp_standard_deviation": 1.0,
        "bp_coefficient_errors": [np.ones(n, dtype=float) for _ in range(n)],
        "rp_coefficient_errors": [np.ones(n, dtype=float) for _ in range(n)],
        "bp_coefficient_correlations": [np.eye(n, dtype=float) for _ in range(n)],
        "rp_coefficient_correlations": [np.eye(n, dtype=float) for _ in range(n)],
    }

    # BP units
    df_bp = pd.DataFrame({
        "source_id": np.arange(n, dtype=np.int64),
        "bp_basis_function_id": int(bp_basis_id),
        "rp_basis_function_id": int(rp_basis_id),
        "bp_coefficients": [I[i] for i in range(n)],
        "rp_coefficients": [np.zeros(n, dtype=float) for _ in range(n)],
        **common,
    })
    df_bp_samp, u = convert(df_bp, sampling=sampling, truncation=False, with_correlation=False, save_file=False)
    df_bp_samp = df_bp_samp[df_bp_samp["xp"].str.upper() == "BP"].sort_values("source_id")
    bp_fluxes = np.vstack([to_1d_float(x) for x in df_bp_samp["flux"]])  # (55,m)
    A_BP = bp_fluxes.T  # (m,55)

    # RP units
    df_rp = pd.DataFrame({
        "source_id": np.arange(n, dtype=np.int64),
        "bp_basis_function_id": int(bp_basis_id),
        "rp_basis_function_id": int(rp_basis_id),
        "bp_coefficients": [np.zeros(n, dtype=float) for _ in range(n)],
        "rp_coefficients": [I[i] for i in range(n)],
        **common,
    })
    df_rp_samp, u2 = convert(df_rp, sampling=sampling, truncation=False, with_correlation=False, save_file=False)
    df_rp_samp = df_rp_samp[df_rp_samp["xp"].str.upper() == "RP"].sort_values("source_id")
    rp_fluxes = np.vstack([to_1d_float(x) for x in df_rp_samp["flux"]])  # (55,m)
    A_RP = rp_fluxes.T

    if not np.allclose(u, u2):
        raise RuntimeError("Sampling grids differ between BP/RP (unexpected).")

    return A_BP.astype(float), A_RP.astype(float), np.asarray(u, float)

## Linear algebra utilities

In [16]:
def ridge_projector(A: np.ndarray, lam: float) -> np.ndarray:
    # A: (m,55) -> returns P: (55,m)
    n = A.shape[1]
    return np.linalg.solve(A.T @ A + lam*np.eye(n), A.T)

def batch_flux(C: np.ndarray, A: np.ndarray) -> np.ndarray:
    # C: (N,55), A: (m,55) -> flux: (N,m)
    return C @ A.T

def batch_derivative(F: np.ndarray, u: np.ndarray) -> np.ndarray:
    # F: (N,m)
    return np.gradient(F, u, axis=1)

def l2_normalize_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(X, axis=1, keepdims=True)
    n = np.maximum(n, eps)
    return X / n

## Query XP continuous coefficients for a chunk of source_ids

In [17]:
def _fetch_xp_for_ids(ids: list[int]) -> pd.DataFrame:
    ids_sql = ",".join(str(int(x)) for x in ids)
    q = f'''
    SELECT
      source_id,
      bp_basis_function_id, rp_basis_function_id,
      bp_coefficients, rp_coefficients
    FROM gaiadr3.xp_continuous_mean_spectrum
    WHERE source_id IN ({ids_sql})
    '''
    job_url = tap_async_start(sess, q, runid="xp_ids_chunk")
    tap_async_wait(sess, job_url, timeout_s=300)
    return tap_async_download_votable(sess, job_url)

def fetch_xp_for_ids(ids, *, min_chunk: int = 200, max_split_depth: int = 6) -> pd.DataFrame:
    ids_list = [int(x) for x in ids]
    try:
        return _fetch_xp_for_ids(ids_list)
    except RuntimeError as e:
        msg = str(e)
        transient = ("HTTP 429", "HTTP 500", "HTTP 502", "HTTP 503", "HTTP 504", "timed out")
        if any(t in msg for t in transient) and max_split_depth > 0 and len(ids_list) > min_chunk:
            mid = len(ids_list) // 2
            left = fetch_xp_for_ids(ids_list[:mid], min_chunk=min_chunk, max_split_depth=max_split_depth - 1)
            right = fetch_xp_for_ids(ids_list[mid:], min_chunk=min_chunk, max_split_depth=max_split_depth - 1)
            return pd.concat([left, right], ignore_index=True)
        raise

## Main loop: compute 110D derivative features and merge into the CSV

In [ ]:
d_cols = [f"d{i:03d}" for i in range(110)]
CACHE = {}  # (bp_basis_id, rp_basis_id) -> (A_BP, A_RP, u, P_BP, P_RP)

def process_df_to_der110(df: pd.DataFrame) -> pd.DataFrame:
    outs = []
    for (bp_id, rp_id), g in df.groupby(["bp_basis_function_id", "rp_basis_function_id"], sort=False):
        key = (int(bp_id), int(rp_id))
        if key not in CACHE:
            A_BP, A_RP, u = build_design_matrices_via_convert(key[0], key[1], SAMPLING)
            P_BP = ridge_projector(A_BP, RIDGE_LAM)
            P_RP = ridge_projector(A_RP, RIDGE_LAM)
            CACHE[key] = (A_BP, A_RP, u, P_BP, P_RP)

        A_BP, A_RP, u, P_BP, P_RP = CACHE[key]

        sids = g["source_id"].astype(np.int64).to_numpy()
        C_bp = np.vstack([to_1d_float(x, 55) for x in g["bp_coefficients"]])
        C_rp = np.vstack([to_1d_float(x, 55) for x in g["rp_coefficients"]])

        F_bp = batch_flux(C_bp, A_BP)
        F_rp = batch_flux(C_rp, A_RP)

        d_bp = batch_derivative(F_bp, u)
        d_rp = batch_derivative(F_rp, u)

        Cbp_prime = d_bp @ P_BP.T  # (N,55)
        Crp_prime = d_rp @ P_RP.T  # (N,55)

        D = np.hstack([Cbp_prime, Crp_prime])          # (N,110)
        D = l2_normalize_rows(D).astype(np.float32)    # L2-normalized features

        out = pd.DataFrame({SOURCE_ID_COL: sids})
        out[d_cols] = D
        outs.append(out)

    if not outs:
        return pd.DataFrame(columns=[SOURCE_ID_COL] + d_cols)

    out_all = pd.concat(outs, ignore_index=True)
    out_all = out_all.drop_duplicates(SOURCE_ID_COL)
    return out_all

parts = []
total_found = 0

for i0 in range(0, len(source_ids), ID_CHUNK_SIZE):
    chunk = source_ids[i0:i0 + ID_CHUNK_SIZE]
    print(f"Chunk {i0//ID_CHUNK_SIZE+1}: ids={len(chunk)}")

    df_xp = fetch_xp_for_ids(chunk)
    if len(df_xp) == 0:
        print("  No XP rows returned for this chunk (no XP available for these source_ids).")
        continue

    df_der110 = process_df_to_der110(df_xp)
    parts.append(df_der110)
    total_found += len(df_der110)
    print("  Derivative feature rows:", len(df_der110), "| total so far:", total_found)

if not parts:
    raise RuntimeError("No XP continuous rows returned for any chunk. Check token / classification IDs / availability of XP.")

df_der110_all = pd.concat(parts, ignore_index=True).drop_duplicates(SOURCE_ID_COL)
print("Total derivative feature rows:", len(df_der110_all))

# Merge into the classification CSV (keep all original rows)
df_merged = df_cls.merge(df_der110_all, on=SOURCE_ID_COL, how="left")

coverage = df_merged[d_cols[0]].notna().mean() * 100.0
print(f"Coverage: {coverage:.1f}% of classification rows got derivative features")

df_merged.to_csv(MERGED_CSV, index=False)
print("Saved merged CSV:", MERGED_CSV)

df_merged.head()

## Quick coverage check

In [ ]:
missing = df_merged[d_cols[0]].isna().sum()
print("Rows without derivative features:", int(missing), "/", len(df_merged))